In [2]:
import numpy as np
import pandas as pd
import scanpy as sc

import os, sys
from sklearn.metrics import mean_squared_error

import warnings 
warnings.filterwarnings('ignore') 

sys.path.append("../../")
import stan

## Loading scRNA dataset
The Processed scRNA-seq data from [this paper](https://www.nature.com/articles/s41588-021-00911-1) is available through the Gene Expression Omnibus under accession number [GSE176078](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE176078).

In [3]:
fname = 'results_breast/breast_scRNA.h5ad'
if os.path.isfile(fname):
    adata_scRNA = sc.read_h5ad(fname)
else:
    path = 'data/Wu2021sc/'
    adata_scRNA = sc.read_mtx(path+"count_matrix_sparse.mtx").transpose()
    adata_scRNA.obs = pd.read_csv(path+"metadata.csv", index_col=0)
    adata_scRNA.var_names = pd.read_csv(path+"count_matrix_genes.tsv", index_col=0, header=None).index.to_list()
    adata_scRNA

    !mkdir results_breast 
    adata_scRNA.write_h5ad("fname")

adata_scRNA

AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'

In [4]:
adatas = dict()
sample_list = ['CID4290A', 'CID4465', 'CID4535', 'CID44971']
for sample in sample_list:
    adatas[sample] = adata_scRNA[adata_scRNA.obs['orig.ident']==sample]
adatas

{'CID4290A': View of AnnData object with n_obs × n_vars = 5789 × 29733
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major',
 'CID4465': View of AnnData object with n_obs × n_vars = 1564 × 29733
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major',
 'CID4535': View of AnnData object with n_obs × n_vars = 3961 × 29733
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major',
 'CID44971': View of AnnData object with n_obs × n_vars = 7986 × 29733
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'}

In [5]:
for sample in sample_list:
    sc.pp.filter_genes(adatas[sample], min_cells=3)
    sc.pp.filter_cells(adatas[sample], min_genes=200)

    adatas[sample].layers['raw'] = adatas[sample].X
    adatas[sample].obs['ncounts'] = adatas[sample].to_df('raw').T.sum()

## Loading the gene-TF prior matrix

In [7]:
for sample in sample_list:
    print(sample)
    adatas[sample] = stan.add_gene_tf_matrix(adatas[sample],
                                            min_cells_proportion = 0.1,
                                            min_tfs_per_gene= 5,
                                            min_genes_per_tf= 5,
                                            gene_tf_source="hTFtarget",
                                            tf_list="humantfs",
                                            source_dir="../data/gene_tf/")
    
    D = adatas[sample].varm['gene_tf']
    Y = adatas[sample].to_df()
    print('gene-cell matrix: {} x {}'.format(Y.shape[1], Y.shape[0]))
    print('gene-TF matrix: {} x {}'.format(D.shape[0], D.shape[1]))
    print('min cells per gene: {}'.format((Y>0).sum().min()))
    print('min genes per cell: {}'.format((Y>0).T.sum().min()))
    print('min tfs per gene: {}'.format(D.T.abs().sum().min()))
    print('min genes per tf: {}'.format(D.abs().sum().min()))

CID4290A
gene-cell matrix: 5076 x 5789
gene-TF matrix: 5076 x 226
min cells per gene: 579
min genes per cell: 140
min tfs per gene: 5
min genes per tf: 6
CID4465
gene-cell matrix: 1250 x 1564
gene-TF matrix: 1250 x 180
min cells per gene: 157
min genes per cell: 97
min tfs per gene: 5
min genes per tf: 5
CID4535
gene-cell matrix: 7160 x 3961
gene-TF matrix: 7160 x 245
min cells per gene: 397
min genes per cell: 177
min tfs per gene: 5
min genes per tf: 5
CID44971
gene-cell matrix: 5312 x 7986
gene-TF matrix: 5312 x 237
min cells per gene: 799
min genes per cell: 171
min tfs per gene: 5
min genes per tf: 5


In [8]:
for sample in sample_list:
    sc.pp.normalize_total(adatas[sample])
    adatas[sample].layers['scaled'] = np.sqrt(adatas[sample].to_df())

## TF activity inference

In [10]:
for sample in sample_list:
    stan.assign_folds(adatas[sample], n_folds=10, random_seed=0)

In [11]:
ridge_models = dict()
for sample in sample_list:
    print(sample)
    ridge_model = stan.Ridge(adatas[sample], layer='scaled')
    ridge_model.fit(n_steps=5, stages=1,
                    grid_search_params={'lam':[1e-4, 1e4]})
    print(ridge_model.params)
    ridge_models[sample] = ridge_model

CID4290A
Time elapsed: 29.30 seconds
{'lam': 0.0001}
CID4465
Time elapsed: 5.45 seconds
{'lam': 0.0001}
CID4535
Time elapsed: 25.39 seconds
{'lam': 1.0}
CID44971
Time elapsed: 43.93 seconds
{'lam': 0.0001}


In [12]:
for sample in sample_list:
    print(sample)
    ridge_model = ridge_models[sample]
    cor, gene_cor = ridge_model.evaluate(fold=-1)
    print("Spot-wise correlation:" + str(round(np.nanmedian(cor), 4)))
    print("Gene-wise correlation: " + str(round(np.nanmedian(gene_cor), 4)))
    
    adatas[sample].obs['pred_cor_ridge'] = cor
    adatas[sample].var['pred_cor_ridge'] = gene_cor
    adatas[sample].obsm['tfa_ridge'] = pd.DataFrame(ridge_model.W_concat.T, 
                                                    index=adatas[sample].obs_names, 
                                                    columns=adatas[sample].uns['tf_names'])

CID4290A
Spot-wise correlation:0.3166
Gene-wise correlation: 0.1175
CID4465
Spot-wise correlation:0.0869
Gene-wise correlation: 0.0628
CID4535
Spot-wise correlation:0.1001
Gene-wise correlation: 0.2292
CID44971
Spot-wise correlation:0.2351
Gene-wise correlation: 0.1108


In [13]:
for sample in sample_list:
    ridge_model = ridge_models[sample]
    Y = adatas[sample].varm['gene_tf'].dot(ridge_model.W_concat)
    mean_squared_error(Y, adatas[sample].to_df('scaled').T)

In [14]:
!mkdir results_breast/breast_sc_ridge
for sample in sample_list:
    adatas[sample].write("results_breast/breast_sc_ridge/{}.h5ad".format(sample))
    

mkdir: results_breast/breast_sc_ridge: File exists


In [15]:
!mv results_breast/breast_sc_ridge/CID4290A.h5ad results_breast/breast_sc_ridge/CID4290.h5ad